# LIBRAIRIES

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
from affine import Affine
from scipy.ndimage import gaussian_filter
from rasterio.features import rasterize, geometry_mask
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.patches as mpatches
import fiona
import rasterio
from rasterio.enums import MergeAlg
from sklearn.preprocessing import MinMaxScaler
from shapely.geometry import Point
import libpysal
import esda
import rasterstats
import warnings
warnings.filterwarnings("ignore")
import math
from statsmodels.nonparametric.smoothers_lowess import lowess
import matplotlib.cm as cm
from IPython.display import display, HTML

import sys
import os
sys.path.insert(0, os.path.abspath("../.."))

from utils.config import *
from utils.functions import *

# ── Force reload ────────────────────────────────────────
import importlib
import utils.config as cfg
import utils.functions as fn
importlib.reload(cfg)
importlib.reload(fn)

# PATH

In [ ]:
operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

# ─── Paths ──────────────────────────────────────────────────────────────────
input_file_path_PL = "/Volumes/T7_lin_win/PANEL_LEMANIQUE/GPS_tracking_data_PL/INPUT/"
output_file_path_PL = "/Volumes/T7_lin_win/PANEL_LEMANIQUE/GPS_tracking_data_PL/OUTPUT/"
input_file_path_PL_wave1 = "/Volumes/T7_lin_win/PANEL_LEMANIQUE/WAVE1_MOBILITY/INPUT/"
output_file_path_PL_wave1 = "/Volumes/T7_lin_win/PANEL_LEMANIQUE/WAVE1_MOBILITY/OUTPUT/"
output_file_path_PL_RASTER = "/Volumes/T7_lin_win/PANEL_LEMANIQUE/GPS_tracking_data_PL/OUTPUT/RASTER/"

input_file_path = '../../Data/input'
output_step1_path='../../Data/output/GE/step-1/'
output_step2_path='../../Data/output/GE/step-2/'
output_step3_path='../../Data/output/GE/step-3/'

# IMPORTS

In [ ]:
canton_GE  = gpd.read_file(f'{input_file_path}/network_agreg/CANTON_GE/CANTON_POLYGON.shp')
canton_GE = canton_GE.to_crs(operation_crs)

legs_GE_walk_regular = gpd.read_parquet(f'{output_file_path_PL}legs_GE_walk_regular.parquet')
communes  = gpd.read_file(input_file_path_PL + "CAD_COMMUNE-SHP/CAD_COMMUNES_GE_fusionnee.shp").to_crs(epsg=2056)
lac_leman = gpd.read_file(input_file_path_PL + "LAC_LEMAN_WITHOUT_BRIDGE-SHP/LAC_LEMAN_WITHOUT_BRIDGE.shp").to_crs(epsg=2056)


# ── Reload zones avec densité et walk_index agrégés ───────────────────────
zones_girec_density  = gpd.read_parquet(f'{output_file_path_PL}/girec_aggregated_index_density.parquet')
carreau_200_density  = gpd.read_parquet(f'{output_file_path_PL}/carreau_200_aggregated_index_density.parquet')

# Reprojection en système de coordonnées opérationnel
zones_girec_density  = zones_girec_density.to_crs(operation_crs)
carreau_200_density  = carreau_200_density.to_crs(operation_crs)

print(f"zones_girec  : {len(zones_girec_density)} zones  | colonnes : {len(zones_girec_density.columns)}")
print(f"carreau_200  : {len(carreau_200_density)} zones | colonnes : {len(carreau_200_density.columns)}")

In [ ]:
index_walkability = gpd.read_parquet(f'{output_step3_path}/step3_index.parquet')
index_walkability = index_walkability.to_crs(operation_crs)

In [ ]:
print(index_walkability.dtypes.to_string())

In [ ]:
attributs_info = pd.read_excel(f"{input_file_path}/attributs/attributs_info.xlsx", sheet_name="attributs_info")

In [ ]:
# ─── Définition de gdf (legs avec géométrie valide) ──────────────────────────
gdf = legs_GE_walk_regular[legs_GE_walk_regular.geometry.notna()].copy()
print(f"gdf : {len(gdf)} legs | {gdf['user_id_fors'].nunique()} utilisateurs uniques")

In [ ]:
# ─── Chargement des rasters exportés ─────────────────────────────────────────
raster_mean_clipped_v2 = load_raster(output_file_path_PL_RASTER, "density_all")

rasters_gender = {g: load_raster(output_file_path_PL_RASTER, f"density_{g}")
                  for g, _ in gender_filters}

rasters_age    = {a: load_raster(output_file_path_PL_RASTER, f"density_{a}")
                  for a, _ in age_filters}

rasters_income = {i: load_raster(output_file_path_PL_RASTER, f"density_{i}")
                  for i, _ in income_filters}

rasters_car    = {c: load_raster(output_file_path_PL_RASTER, f"density_{c}")
                  for c, _ in car_filters}

rasters_tp     = {t: load_raster(output_file_path_PL_RASTER, f"density_{t}")
                  for t, _ in tp_filters}

rasters_hourly = {s: load_raster(output_file_path_PL_RASTER, f"density_{s}")
                  for s, _, _, _, _ in time_slots}

# ─── Vérification ─────────────────────────────────────────────────────────────
print(f"✓ raster_mean_clipped_v2 : {raster_mean_clipped_v2.shape}")
print(f"✓ rasters_gender         : {list(rasters_gender.keys())}")
print(f"✓ rasters_age            : {list(rasters_age.keys())}")
print(f"✓ rasters_income         : {list(rasters_income.keys())}")
print(f"✓ rasters_car            : {list(rasters_car.keys())}")
print(f"✓ rasters_tp             : {list(rasters_tp.keys())}")
print(f"✓ rasters_hourly         : {list(rasters_hourly.keys())}")

In [ ]:
# ─── Chargement des rasters horaires ─────────────────────────────────────────
hourly_slots = [("all_day", 0, 0, 23, 59)] + [
    (f"{h:02d}h", h, 0, h+1, 0) for h in range(6, 22)
]
hourly_names = [s for s, *_ in hourly_slots if s != "all_day"]
hourly_labels = {f"{h:02d}h": f"{h:02d}h - {h+1:02d}h" for h in range(6, 22)}
hourly_labels["all_day"] = "All day"

rasters_hourly_h = {s: load_raster(output_file_path_PL_RASTER, f"density_hourly_{s}")
                    for s in hourly_names + ["all_day"]}

# ─── Normalisation commune ────────────────────────────────────────────────────
hourly_h_max = compute_scale(
    [rasters_hourly_h[s] for s in hourly_names if (rasters_hourly_h[s] > 0).any()],
    clip_percentile, mode=norm_mode
)

print(f"✓ {len(rasters_hourly_h)} rasters horaires chargés")
print(f"hourly_h_max : {hourly_h_max:.6f}")

In [ ]:
# ─── Reprojection en EPSG:2056 ────────────────────────────────────────────────
legs_GE_walk_regular = legs_GE_walk_regular.to_crs(epsg=2056)
canton_GE            = canton_GE.to_crs(epsg=2056)

print(f"CRS legs   : {legs_GE_walk_regular.crs}")
print(f"CRS canton : {canton_GE.crs}")

# ─── Paramètres raster ────────────────────────────────────────────────────────
pixel_size = 10

xmin, ymin, xmax, ymax = canton_GE.total_bounds

raster_width  = int((xmax - xmin) / pixel_size)
raster_height = int((ymax - ymin) / pixel_size)

transform = Affine(pixel_size, 0, xmin,
                   0, -pixel_size, ymax)

print(f"total_bounds  : {xmin:.0f}, {ymin:.0f}, {xmax:.0f}, {ymax:.0f}")
print(f"raster_width  : {raster_width}")
print(f"raster_height : {raster_height}")

# ─── Masque canton ────────────────────────────────────────────────────────────
canton_mask = geometry_mask(
    geometries=canton_GE.geometry,
    out_shape=(raster_height, raster_width),
    transform=transform,
    invert=True
)

lac_mask = geometry_mask(
    geometries=lac_leman.geometry,
    out_shape=(raster_height, raster_width),
    transform=transform,
    invert=True
)

canton_GE_mask = canton_mask & ~lac_mask

print(f"pixels canton  : {canton_GE_mask.sum():,}")

In [ ]:
# ─── Extent ───────────────────────────────────────────────────────────────────
extent = [xmin, xmax, ymin, ymax]  

# ─── Girec ────────────────────────────────────────────────────────────────────
girec = gpd.read_file(input_file_path_PL + "GEO_GIREC-SHP/GEO_GIREC.shp").to_crs(epsg=2056)

# ─── Focus commune ────────────────────────────────────────────────────────────
focus_commune_name = "Genève"
focus_commune      = communes[communes["COMMUNE"] == focus_commune_name]
xmin_z, ymin_z, xmax_z, ymax_z = focus_commune.total_bounds
margin = 200

In [ ]:
print(carreau_200_density.dtypes.to_string())

In [ ]:
# ═══ Affichage des rasters chargés — normalisation par groupe ═════════════════

clip_percentile = 99
norm_mode       = "linear"

def plot_group_rasters(rasters_dict, filters_list, group_label,
                       gdf_group=None, group_col=None, group_labels=None,
                       filters_contribution=None):  # ← nouveau paramètre

    group_max = compute_scale(
        [rasters_dict[k] for k, _ in filters_list if k in rasters_dict],
        clip_percentile, mode=norm_mode
    )
    print(f"{group_label} max : {group_max:.6f}")

    for key, _ in filters_list:
        if key not in rasters_dict:
            print(f"  ⚠ clé manquante : {key}")
            continue

        label        = labels_all_groups[key]
        density_norm = normalize_raster(rasters_dict[key], group_max, mode=norm_mode)

        print(f"─── {group_label} — {label} — vue canton complet ───")
        plot_density_map(
            density_norm,
            title=f"",
            extent=extent, canton_GE=canton_GE, girec=girec, clip_percentile_label=99,
        )

        print(f"─── {group_label} — {label} — vue zoomée ───")
        plot_density_map(
            density_norm,
            title=f"",
            extent=extent, canton_GE=canton_GE, girec=girec,
            focus=focus_commune,
            zoom_bounds=(xmin_z, ymin_z, xmax_z, ymax_z),
            margin=margin,
            clip_percentile_label=99,
        )
        del density_norm

    # ─── Plot de contribution ─────────────────────────────────────────────────
    if gdf_group is not None and group_col is not None and group_labels is not None:
        _filters = filters_contribution if filters_contribution is not None else filters_list
        print(f"─── {group_label} — contribution plot ───")
        plot_user_contribution(
            gdf_group     = gdf_group,
            group_filters = _filters,
            group_labels  = group_labels,
            group_col     = group_col,
            title         = f"" #f"User contribution — {group_label}"
        )


In [ ]:
# ─── 1. Raster global ─────────────────────────────────────────────────────────
global_max   = compute_scale([raster_mean_clipped_v2], clip_percentile, mode=norm_mode)
density_norm = normalize_raster(raster_mean_clipped_v2, global_max, mode=norm_mode)

print(f"─── Global — All users — vue canton complet ───")
plot_density_map(
    density_norm,
    title=f"", #f"Mean Daily Pedestrian Density — All users",
    extent=extent, canton_GE=canton_GE, girec=girec, clip_percentile_label=99
)

print(f"─── Global — All users — vue zoomée ───")
plot_density_map(
    density_norm,
    title=f"", #f"Mean Daily Pedestrian Density — Zoom {focus_commune_name}\nAll users",
    extent=extent, canton_GE=canton_GE, girec=girec,
    focus=focus_commune,
    zoom_bounds=(xmin_z, ymin_z, xmax_z, ymax_z),
    margin=margin,
    clip_percentile_label=99
)
del density_norm

plot_user_contribution(
    gdf_group     = gdf,
    group_filters = [("all", None)],
    group_labels  = {"all": labels_all_groups["all"]},  # → "All users"
    group_col     = "tp_level",  # ← peu importe la colonne ici, puisqu'on ne filtre pas
    title         = ""
)

In [ ]:
# ─── 2. Genre ─────────────────────────────────────────────────────────────────
plot_group_rasters(rasters_gender, gender_filters, "Gender",
                   gdf_group=gdf, group_col="gdr",
                   group_labels=gender_labels,
                   filters_contribution=gender_filters)  # ← valeurs françaises

In [ ]:
# ─── 3. Âge ───────────────────────────────────────────────────────────────────
plot_group_rasters(rasters_age, age_filters, "Age",
                   gdf_group=gdf, group_col="age_fr_grouped",
                   group_labels=age_labels,
                   filters_contribution=age_filters)  # ← valeurs françaises

In [ ]:
# ─── 4. Revenu ────────────────────────────────────────────────────────────────
plot_group_rasters(rasters_income, income_filters, "Income",
                   gdf_group=gdf, group_col="income_class",
                   group_labels=income_labels,
                   filters_contribution=income_filters_en)  # ← valeurs anglaises

In [ ]:
# ─── 5. Voiture ───────────────────────────────────────────────────────────────
plot_group_rasters(rasters_car, car_filters, "Car ownership",
                   gdf_group=gdf, group_col="has_car",
                   group_labels=car_labels,
                   filters_contribution=car_filters_en)  # ← valeurs anglaises

In [ ]:
# ─── 6. Abonnement TP ─────────────────────────────────────────────────────────
plot_group_rasters(rasters_tp, tp_filters, "PT subscription",
                   gdf_group=gdf, group_col="tp_level",
                   group_labels=tp_labels)

In [ ]:
plot_small_multiples(
    rasters_dict    = rasters_hourly_h,
    slot_names      = hourly_names,
    labels_dict     = hourly_labels,
    title           = "", #"Mean Daily Pedestrian Density — Hourly | All users",
    extent          = extent,
    canton_GE       = canton_GE,
    girec           = girec,
    d_max           = hourly_h_max,
    ncols           = 4,
    plot_individual = False,
    subtitle_fontsize = 14,   # ← seulement ici, parce que 16 panneaux
    cbar_fontsize=14,
)

In [ ]:
plot_small_multiples(
    rasters_dict    = rasters_hourly_h,
    slot_names      = hourly_names,
    labels_dict     = hourly_labels,
    title           =  f"", #f"Mean Daily Pedestrian Density — Hourly | Zoom {focus_commune_name}",
    extent          = extent,
    canton_GE       = canton_GE,
    girec           = girec,
    d_max           = hourly_h_max,
    zoom_bounds     = (xmin_z, ymin_z, xmax_z, ymax_z),
    margin          = margin,
    ncols           = 4,
    plot_individual = False,
    subtitle_fontsize = 14,   # ← seulement ici, parce que 16 panneaux
    cbar_fontsize=14,
)

In [ ]:
# ─── Normalisation spéciale P100 pour cette carte ────────────────────────────
global_max_p100   = compute_scale([raster_mean_clipped_v2], 100, mode=norm_mode)
density_norm_p100 = normalize_raster(raster_mean_clipped_v2, global_max_p100, mode=norm_mode)

print(f"─── Global — All users — vue canton complet (P100) ───")
plot_density_map(
    density_norm_p100,
    title=f"",
    extent=extent, canton_GE=canton_GE, girec=girec,
    clip_percentile_label=100
)

print(f"─── Global — All users — vue zoomée (P100) ───")
plot_density_map(
    density_norm_p100,
    title=f"",
    extent=extent, canton_GE=canton_GE, girec=girec,
    focus=focus_commune,
    zoom_bounds=(xmin_z, ymin_z, xmax_z, ymax_z),
    margin=margin,
    clip_percentile_label=100
)
del density_norm_p100

# CARREAUX_200

In [ ]:
# ── Étape 1 : Préparation des données ─────────────────────────────────────

# 1. Extraire la liste des attributs à analyser
attrs = attributs_info[attributs_info['include_in_index'] == True]['attribute'].tolist()
print(f"{'='*50}")
print(f"Attributs à analyser : {len(attrs)}")
print(f"{'='*50}")
print(attrs)

# 2. Extraire les colonnes de densité disponibles
density_cols = [c for c in carreau_200_density.columns if c.startswith('density_') and not c.endswith('_norm')]
print(f"\n{'='*50}")
print(f"Colonnes de densité disponibles : {len(density_cols)}")
print(f"{'='*50}")
print(density_cols)

# 3. Vérifier que tous les attributs existent dans carreau_200
missing = [a for a in attrs if a not in carreau_200_density.columns]
print(f"\n{'='*50}")
if missing:
    print(f"⚠ Attributs manquants dans carreau_200 ({len(missing)}) :")
    print(missing)
else:
    print(f"✓ Tous les attributs sont présents dans carreau_200")
print(f"{'='*50}")

# 4. Construire le sous-dataframe de travail
cols_to_keep = attrs + density_cols
df_work      = carreau_200_density[cols_to_keep].copy()

print(f"\n{'='*50}")
print(f"Dataframe de travail : {df_work.shape[0]} zones × {df_work.shape[1]} colonnes")
print(f"  → {len(attrs)} attributs + {len(density_cols)} colonnes de densité")
print(f"{'='*50}")

In [ ]:
df_work

In [ ]:
display(HTML(df_work.head(10).to_html()))

In [ ]:
# Variance de chaque attribut dans carreau_200
df_work[attrs].var().sort_values()

In [ ]:
df_work[attrs].std().sort_values()

In [ ]:
# Écart-type population (comme QGIS)
df_work[attrs].std(ddof=0).sort_values()

In [ ]:
variance_threshold = 0.001

variances = df_work[attrs].var().sort_values(ascending=False)

low_variance_attrs = set(variances[variances < variance_threshold].index.tolist())
print(low_variance_attrs)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.bar(variances.index, variances.values, color='steelblue', edgecolor='white', linewidth=0.5)
ax.axhline(variance_threshold, color='red', linewidth=1, linestyle='--',
           label=f'Threshold = {variance_threshold}')

for attr, val in variances.items():
    ax.text(attr, val + 0.001, f'{val:.4f}',
            ha='center', va='bottom', fontsize=7, rotation=45)

ax.set_ylabel('Variance')
ax.set_xlabel('Attribute')
ax.set_title(f'Attribute variance across 200x200m grid\nAttributes below threshold (var < {variance_threshold}) are highlighted in red',
             fontsize=11)

# Labels en rouge pour les attributs sous le seuil
for label in ax.get_xticklabels():
    if label.get_text() in low_variance_attrs:
        label.set_color('#d73027')

ax.set_xticklabels(variances.index, rotation=45, ha='right', fontsize=8)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# ALL CANTON PIXELS - CARREAUX_200

## CORRELATION SPEARMAN

### RESULTATS

In [ ]:
# ── Étape 2 : Calcul des corrélations de Spearman ─────────────────────────

results = []

for attr in attrs:
    row = {'attribute': attr}
    
    for density_col in density_cols:
        # Garder uniquement les lignes sans NaN pour les deux colonnes
        mask   = df_work[[attr, density_col]].notna().all(axis=1)
        x      = df_work.loc[mask, attr]
        y      = df_work.loc[mask, density_col]
        
        if len(x) < 10:  # sécurité si trop peu de données
            row[density_col] = np.nan
            continue
        
        r, p        = spearmanr(x, y)
        row[density_col] = round(r, 4)
    
    results.append(row)

df_corr = pd.DataFrame(results).set_index('attribute')

print(f"{'='*50}")
print(f"Tableau de corrélations : {df_corr.shape[0]} attributs × {df_corr.shape[1]} groupes")
print(f"{'='*50}")
display(HTML(df_corr.to_html()))

### PLOT

In [ ]:
# Ordre des colonnes selon group_categories
ordered_cols = []
for cat, cols in group_categories.items():
    existing = [c for c in cols if c in df_corr.columns]
    ordered_cols.extend(existing)

# Dimensions du plot
ncols = max(len([c for c in cols if c in df_corr.columns])
            for cols in group_categories.values()
            if any(c in df_corr.columns for c in cols))
nrows = len([cat for cat, cols in group_categories.items()
             if any(c in df_corr.columns for c in cols)])

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 7 * nrows))
axes = np.array(axes).reshape(nrows, ncols)

row = 0
for cat, cols in group_categories.items():
    existing = [c for c in cols if c in df_corr.columns]
    if not existing:
        continue

    for j, col in enumerate(existing):
        ax        = axes[row, j]
        group_key = col.replace('density_', '')
        values    = df_corr.loc[attrs[::-1], col]
        colors    = ['#d73027' if v < 0 else '#1a9850' for v in values]

        ax.barh(values.index, values.values, color=colors, edgecolor='white', linewidth=0.5)
        ax.axvline(0, color='black', linewidth=0.8)
        ax.set_xlim(-1, 1)
        ax.set_title(labels_all_groups.get(group_key, group_key), fontsize=9)
        ax.spines[["top", "right"]].set_visible(False)

        # ── Surbrillance ligne entière ─────────────────────────────────────
        yticks = list(values.index)
        for k, attr in enumerate(yticks):
            if attr in low_variance_attrs:
                ax.axhspan(k - 0.5, k + 0.5, color='#ffcccc', alpha=0.4, zorder=0)

        # ── Surbrillance labels ────────────────────────────────────────────
        for label in ax.get_yticklabels():
            if label.get_text() in low_variance_attrs:
                label.set_backgroundcolor('#ffcccc')
                label.set_color('#d73027')

        for attr, val in values.items():
            if val < 0:
                ax.text(val - 0.02, attr, f'{val:.3f}',
                        ha='right', va='center', fontsize=6, color='#d73027')
            else:
                ax.text(val + 0.02, attr, f'{val:.3f}',
                        ha='left', va='center', fontsize=6, color='#1a9850')

        # Cacher les y labels sauf pour la première colonne
        if j > 0:
            ax.set_yticklabels([])

    # Cacher les axes vides sur la ligne
    for j in range(len(existing), ncols):
        axes[row, j].set_axis_off()

    # Label catégorie sur le côté
    axes[row, 0].set_ylabel(cat.upper(), fontsize=10, fontweight='bold', labelpad=10)

    row += 1

plt.suptitle(f'Spearman correlation — density × attributes | Carreau 200m\n'
             f'⚠ Red background = low variance attributes (var < {variance_threshold})',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
ncols = 4
nrows = math.ceil(len(attrs) / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = axes.flatten()

for i, attr in enumerate(attrs):
    ax     = axes[i]
    values = df_corr.loc[attr]  # une ligne = tous les profils pour cet attribut
    colors = ['#d73027' if v < 0 else '#1a9850' for v in values]

    ax.bar(
        [labels_all_groups.get(c.replace('density_', ''), c.replace('density_', '')) for c in values.index],
        values.values,
        color=colors, edgecolor='white', linewidth=0.5
    )
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylim(-1, 1)
    ax.set_title(attr, fontsize=9,
                 color='#d73027' if attr in low_variance_attrs else 'black')
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xticklabels(
        [labels_all_groups.get(c.replace('density_', ''), c.replace('density_', '')) for c in values.index],
        rotation=45, ha='right', fontsize=6
    )

    for col, val in zip(values.index, values.values):
        label = labels_all_groups.get(col.replace('density_', ''), col.replace('density_', ''))
        if val < 0:
            ax.text(label, val - 0.02, f'{val:.4f}',
                    ha='center', va='top', fontsize=5, color='#d73027')
        else:
            ax.text(label, val + 0.02, f'{val:.4f}',
                    ha='center', va='bottom', fontsize=5, color='#1a9850')

    # Surbrillance titre si low variance
    if attr in low_variance_attrs:
        ax.set_facecolor('#fff5f5')

for j in range(i + 1, len(axes)):
    axes[j].set_axis_off()

plt.suptitle(f'Spearman correlation by attribute — all profiles | Carreau 200m\n'
             f'⚠ Red title = low variance attribute (var < {variance_threshold})',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## DENSITY-WEIGHTED ATTR MEANS

### RESULTS

In [ ]:
# ── Étape 1 : Moyenne cantonale brute de chaque attribut ──────────────────

canton_means = df_work[attrs].mean()

print(f"{'='*50}")
print(f"Canton average per attribute")
print(f"{'='*50}")
for attr, val in canton_means.items():
    print(f"  {attr:<25} : {val:.4f}")
print(f"{'='*50}")

In [ ]:
# ── Étape 2 : Moyenne pondérée par densité pour chaque groupe ─────────────

results_weighted = []

for attr in attrs:
    row = {'attribute': attr}
    
    for density_col in density_cols:
        # Garder uniquement les lignes sans NaN pour les deux colonnes
        mask   = df_work[[attr, density_col]].notna().all(axis=1)
        x      = df_work.loc[mask, attr]
        w      = df_work.loc[mask, density_col]
        
        if w.sum() == 0:
            row[density_col] = np.nan
            continue
        
        weighted_mean    = np.average(x, weights=w)
        row[density_col] = round(weighted_mean, 4)
    
    results_weighted.append(row)

df_weighted = pd.DataFrame(results_weighted).set_index('attribute')

print(f"{'='*50}")
print(f"Density-weighted means : {df_weighted.shape[0]} attributs × {df_weighted.shape[1]} groupes")
print(f"{'='*50}")
display(HTML(df_weighted.to_html()))

In [ ]:
# ── Étape 3 : Delta vs moyenne cantonale ──────────────────────────────────

df_delta_weighted = df_weighted.subtract(canton_means, axis=0).round(4)

print(f"{'='*50}")
print(f"Delta — density-weighted mean vs canton average")
print(f"{'='*50}")
display(HTML(df_delta_weighted.to_html()))

### PLOT

In [ ]:
ncols = max(len([c for c in cols if c in df_delta_weighted.columns])
            for cols in group_categories.values()
            if any(c in df_delta_weighted.columns for c in cols))
nrows = len([cat for cat, cols in group_categories.items()
             if any(c in df_delta_weighted.columns for c in cols)])

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 7 * nrows))
axes = np.array(axes).reshape(nrows, ncols)

row = 0
for cat, cols in group_categories.items():
    existing = [c for c in cols if c in df_delta_weighted.columns]
    if not existing:
        continue

    for j, col in enumerate(existing):
        ax        = axes[row, j]
        group_key = col.replace('density_', '')
        values    = df_delta_weighted.loc[attrs[::-1], col]
        colors    = ['#d73027' if v < 0 else '#1a9850' for v in values]

        ax.barh(values.index, values.values, color=colors, edgecolor='white', linewidth=0.5)
        ax.axvline(0, color='black', linewidth=0.8)
        ax.set_xlim(-0.5, 0.5)
        ax.set_title(labels_all_groups.get(group_key, group_key), fontsize=9)
        ax.spines[["top", "right"]].set_visible(False)

        # ── Surbrillance ligne entière ─────────────────────────────────────
        yticks = list(values.index)
        for k, attr in enumerate(yticks):
            if attr in low_variance_attrs:
                ax.axhspan(k - 0.5, k + 0.5, color='#ffcccc', alpha=0.4, zorder=0)

        # ── Surbrillance labels ────────────────────────────────────────────
        for label in ax.get_yticklabels():
            if label.get_text() in low_variance_attrs:
                label.set_backgroundcolor('#ffcccc')
                label.set_color('#d73027')

        for attr, val in values.items():
            if val < 0:
                ax.text(val - 0.01, attr, f'{val:.3f}',
                        ha='right', va='center', fontsize=6, color='#d73027')
            else:
                ax.text(val + 0.01, attr, f'{val:.3f}',
                        ha='left', va='center', fontsize=6, color='#1a9850')

        if j > 0:
            ax.set_yticklabels([])

    for j in range(len(existing), ncols):
        axes[row, j].set_axis_off()

    axes[row, 0].set_ylabel(cat.upper(), fontsize=10, fontweight='bold', labelpad=10)

    row += 1

plt.suptitle(f'Δ Density-weighted mean vs canton average | Carreau 200m\n'
             f'⚠ Red background = low variance attributes (var < {variance_threshold})',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Définition des groupes et leurs tailles ─────────────────────────────────
group_sizes = [
    1, #all user
    len(gender_filters),
    len(age_filters),
    len(income_filters),
    len(car_filters),
    len(tp_filters),
]

ncols = 4
nrows = math.ceil(len(attrs) / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = axes.flatten()

for i, attr in enumerate(attrs):
    ax     = axes[i]
    values = df_delta_weighted.loc[attr]
    colors = ['#d73027' if v < 0 else '#1a9850' for v in values]

    x_labels = [labels_all_groups.get(c.replace('density_', ''), 
                c.replace('density_', '')) for c in values.index]

    ax.bar(x_labels, values.values, color=colors, edgecolor='white', linewidth=0.5)


    # ─── Séparateurs verticaux entre groupes ─────────────────────────────────────
    sep_x = -0.5  # position de départ
    for size in group_sizes[:-1]:  # pas de séparateur après le dernier groupe
        sep_x += size
        ax.axvline(sep_x, color='gray', linewidth=0.6, linestyle='--', alpha=0.5)


    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylim(-0.5, 0.5)
    ax.set_title(attr, fontsize=9,
                 color='#d73027' if attr in low_variance_attrs else 'black')
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=6)

    for label, val in zip(x_labels, values.values):
        if val < 0:
            ax.text(label, val - 0.01, f'{val:.3f}',
                    ha='center', va='top', fontsize=5, color='#d73027')
        else:
            ax.text(label, val + 0.01, f'{val:.3f}',
                    ha='center', va='bottom', fontsize=5, color='#1a9850')

    if attr in low_variance_attrs:
        ax.set_facecolor('#fff5f5')

for j in range(i + 1, len(axes)):
    axes[j].set_axis_off()

plt.suptitle(f'Δ Density-weighted mean vs canton average by attribute | Carreau 200m\n'
             f'⚠ Red title/background = low variance attribute (var < {variance_threshold})',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(16, 10))

# Renommer les colonnes avec les labels lisibles
col_labels = [labels_all_groups.get(c.replace('density_', ''), 
              c.replace('density_', '')) for c in df_delta_weighted.columns]

im = ax.imshow(
    df_delta_weighted.values,
    cmap='RdYlGn',
    aspect='auto',
    vmin=-0.6, vmax=0.6
)

ax.set_xticks(range(len(col_labels)))
ax.set_yticks(range(len(attrs)))
ax.set_xticklabels(col_labels, rotation=35, ha='right', fontsize=9)
ax.set_yticklabels(df_delta_weighted.index, fontsize=9)

# Valeurs dans les cellules
for i in range(len(df_delta_weighted.index)):
    for j in range(len(df_delta_weighted.columns)):
        val = df_delta_weighted.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:+.3f}',
                    ha='center', va='center', fontsize=7,
                    color='white' if abs(val) > 0.35 else 'black')

plt.colorbar(im, ax=ax, fraction=0.02, pad=0.01,
             label='Δ density-weighted mean vs canton average')
ax.set_title('Δ Density-weighted mean vs canton average — Carreau 200m',
             fontsize=12, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

In [ ]:
print(carreau_200_density.dtypes.to_string())

## DELTA MEAN BY AREA_TYPE - CARREAUX

In [ ]:
# ── Vérification AREA_TYPE sur carreau_200 ────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 8))

for area_type, color in AREA_TYPE_COLORS.items():
    mask = carreau_200_density['AREA_TYPE'].str.lower() == area_type
    if mask.sum() > 0:
        carreau_200_density[mask].plot(ax=ax, color=color, linewidth=0, alpha=0.85)

ax.set_title('AREA_TYPE — Carreaux 200m', fontsize=12, fontweight='bold')
ax.set_axis_off()
patches = [mpatches.Patch(color=AREA_TYPE_COLORS[a], label=a.title())
           for a in AREA_TYPE_ORDER]
ax.legend(handles=patches, loc='lower left', fontsize=9, framealpha=0.9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Analyse stratifiée par AREA_TYPE ─────────────────────────────────────

# Ajouter AREA_TYPE au df_work
df_work_area = df_work.copy()
df_work_area['AREA_TYPE'] = carreau_200_density['AREA_TYPE'].str.lower()

results_by_area = {}

for area_type in AREA_TYPE_ORDER:
    df_area = df_work_area[df_work_area['AREA_TYPE'] == area_type].drop(columns='AREA_TYPE')
    
    if len(df_area) == 0:
        continue

    rows = []
    for attr in attrs:
        row = {'attribute': attr}
        for density_col in density_cols:
            mask = df_area[[attr, density_col]].notna().all(axis=1)
            x    = df_area.loc[mask, attr]
            w    = df_area.loc[mask, density_col]
            if w.sum() == 0:
                row[density_col] = np.nan
                continue
            row[density_col] = round(np.average(x, weights=w), 4)
        rows.append(row)

    df_weighted_area = pd.DataFrame(rows).set_index('attribute')
    df_delta_area    = df_weighted_area.subtract(canton_means, axis=0).round(4)

    results_by_area[area_type] = {
        'weighted': df_weighted_area,
        'delta':    df_delta_area,
        'n':        len(df_area)
    }

    print(f"\n{'='*50}")
    print(f"  {area_type.title()} (n={len(df_area)} carreaux)")
    print(f"{'='*50}")
    display(HTML(df_delta_area.to_html()))

In [ ]:
# Extraire un attribut spécifique pour tous les AREA_TYPE et tous les groupes
attr_focus   = 'eclairage'
groups_focus = ['density_no_car', 'density_has_car',
                'density_tres_modeste', 'density_aise']
labels_focus = ['No car', 'Has car', 'Low income', 'High income']

# ── Print ─────────────────────────────────────────────────────────────────
print(f"{'='*60}")
print(f"  Delta density-weighted — {attr_focus}")
print(f"  Référence : moyenne cantonale globale")
print(f"{'='*60}")

col_w  = 14
name_w = 30
header = f"{'AREA_TYPE':<{name_w}}" + "".join(f"{l:>{col_w}}" for l in labels_focus)
print(header)
print("─" * len(header))

for area_type in AREA_TYPE_ORDER:
    if area_type not in results_by_area:
        continue
    df_delta = results_by_area[area_type]['delta']
    n        = results_by_area[area_type]['n']
    row      = f"{area_type.title()+f' (n={n})':<{name_w}}"
    for g in groups_focus:
        val = df_delta.loc[attr_focus, g]
        row += f"{val:>+{col_w}.3f}"
    print(row)

# ── Plot ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(AREA_TYPE_ORDER), figsize=(18, 5), sharey=True)
fig.suptitle(f'Delta density-weighted — {attr_focus}', fontsize=13, fontweight='bold')

for ax, area_type in zip(axes, AREA_TYPE_ORDER):
    if area_type not in results_by_area:
        ax.set_visible(False)
        continue

    df_delta = results_by_area[area_type]['delta']
    n        = results_by_area[area_type]['n']
    vals     = [df_delta.loc[attr_focus, g] for g in groups_focus]
    colors   = ['#5B7FBF', '#95B4D4', '#E8A838', '#fdc980']

    bars = ax.bar(labels_focus, vals, color=colors, edgecolor='white', linewidth=0.5)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(f'{area_type.title()}\n(n={n})', fontsize=9, fontweight='bold')
    ax.tick_params(axis='x', rotation=30)
    ax.grid(axis='y', alpha=0.3)

    for bar, val in zip(bars, vals):
        if not np.isnan(val):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.005 if val >= 0 else bar.get_height() - 0.015,
                    f'{val:+.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
for g in groups_focus:
    mask = (df_work_area['AREA_TYPE'] == 'secondary centers') & df_work_area[g].notna() & (df_work_area[g] > 0)
    print(f"{g:<35} : {mask.sum()} carreaux actifs")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
mask = carreau_200_density['AREA_TYPE'].str.lower() == 'secondary centers'
carreau_200_density[mask].plot(ax=ax, color=AREA_TYPE_COLORS['secondary centers'], linewidth=0)
carreau_200_density[~mask].plot(ax=ax, color='#eeeeee', linewidth=0, alpha=0.5)
ax.set_title('Secondary centers', fontsize=12)
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
col_labels = [labels_all_groups.get(c.replace('density_', ''), 
              c.replace('density_', '')) for c in density_cols]

for area_type in AREA_TYPE_ORDER:
    if area_type not in results_by_area:
        continue

    df_delta_area = results_by_area[area_type]['delta']
    n             = results_by_area[area_type]['n']

    fig, ax = plt.subplots(figsize=(16, 10))

    im = ax.imshow(
        df_delta_area.values,
        cmap='RdYlGn',
        aspect='auto',
        vmin=-0.6, vmax=0.6
    )

    ax.set_xticks(range(len(col_labels)))
    ax.set_yticks(range(len(df_delta_area.index)))
    ax.set_xticklabels(col_labels, rotation=35, ha='right', fontsize=9)
    ax.set_yticklabels(df_delta_area.index, fontsize=9)

    for i in range(len(df_delta_area.index)):
        for j in range(len(df_delta_area.columns)):
            val = df_delta_area.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f'{val:+.3f}',
                        ha='center', va='center', fontsize=7,
                        color='white' if abs(val) > 0.35 else 'black')

    plt.colorbar(im, ax=ax, fraction=0.02, pad=0.01,
                 label='Δ density-weighted mean vs canton average')
    ax.set_title(
        f'Δ Density-weighted mean vs canton average\n'
        f'{area_type.title()} (n={n} carreaux)',
        fontsize=12, fontweight='bold', pad=12
    )
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Paires contrastées ────────────────────────────────────────────────────
contrast_pairs = [
    ('density_femme',        'density_homme',       'Women vs Men'),
    ('density_18-29',        'density_60+',         '18-29 vs 60+'),
    ('density_tres_modeste', 'density_aise',        'Low vs High income'),
    ('density_no_car',       'density_has_car',     'No car vs Car owner'),
    ('density_full_tp',      'density_no_tp',       'Full TP vs No TP'),
]

# ── Calcul des écarts ─────────────────────────────────────────────────────
contrast_labels = [label for _, _, label in contrast_pairs]
df_contrast     = pd.DataFrame(index=df_delta_weighted.index, columns=contrast_labels)

for col_a, col_b, label in contrast_pairs:
    df_contrast[label] = (df_delta_weighted[col_a] - df_delta_weighted[col_b]).round(4)

df_contrast = df_contrast.astype(float)

# ── Heatmap ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 10))

im = ax.imshow(
    df_contrast.values,
    cmap='RdYlGn',
    aspect='auto',
    vmin=-0.15, vmax=0.15
)

ax.set_xticks(range(len(contrast_labels)))
ax.set_yticks(range(len(df_contrast.index)))
ax.set_xticklabels(contrast_labels, rotation=25, ha='right', fontsize=10)
ax.set_yticklabels(df_contrast.index, fontsize=9)

for i in range(len(df_contrast.index)):
    for j in range(len(df_contrast.columns)):
        val = df_contrast.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:+.3f}',
                    ha='center', va='center', fontsize=8,
                    color='white' if abs(val) > 0.08 else 'black')

plt.colorbar(im, ax=ax, fraction=0.02, pad=0.01,
             label='Δ group A − group B (density-weighted mean)')
ax.set_title(
    'Contrast between groups — Δ density-weighted mean\n'
    'Positive = group A walks in better conditions than group B',
    fontsize=12, fontweight='bold', pad=12
)
plt.tight_layout()
plt.show()

# GIREC

In [ ]:
# ══ DENSITY-WEIGHTED MEAN — GIREC SUB-SECTORS ════════════════════════════════

# ── Étape 1 : Préparation des données ────────────────────────────────────────
# 1. Extraire la liste des attributs à analyser
attrs_girec = attributs_info[attributs_info['include_in_index'] == True]['attribute'].tolist()
print(f"{'='*50}")
print(f"Attributs à analyser : {len(attrs_girec)}")
print(f"{'='*50}")
print(attrs_girec)

# 2. Extraire les colonnes de densité disponibles
density_cols_girec = [
    c for c in zones_girec_density.columns 
    if c.startswith('density_') and not c.endswith('_norm')
]
print(f"\n{'='*50}")
print(f"Colonnes de densité disponibles : {len(density_cols_girec)}")
print(f"{'='*50}")
print(density_cols_girec)

# 3. Vérifier que tous les attributs existent dans zones_girec_density
missing_girec = [a for a in attrs_girec if a not in zones_girec_density.columns]
print(f"\n{'='*50}")
if missing_girec:
    print(f"⚠ Attributs manquants dans zones_girec_density ({len(missing_girec)}) :")
    print(missing_girec)
else:
    print(f"✓ Tous les attributs sont présents dans zones_girec_density")
print(f"{'='*50}")

# 4. Construire le sous-dataframe de travail
cols_to_keep_girec = attrs_girec + density_cols_girec
df_work_girec      = zones_girec_density[cols_to_keep_girec].copy()
print(f"\n{'='*50}")
print(f"Dataframe de travail — GIREC sub-sectors")
print(f"  {df_work_girec.shape[0]} zones × {df_work_girec.shape[1]} colonnes")
print(f"  → {len(attrs_girec)} attributs + {len(density_cols_girec)} colonnes de densité")
print(f"{'='*50}")

In [ ]:
df_work_girec[attrs_girec].var().sort_values()

In [ ]:
# ── Variance des attributs ────────────────────────────────────────────────────
variance_threshold_girec = 0.001
variances_girec          = df_work_girec[attrs_girec].var().sort_values(ascending=False)
low_variance_attrs_girec = set(variances_girec[variances_girec < variance_threshold_girec].index.tolist())
print(f"Attributs sous le seuil : {low_variance_attrs_girec}")

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_alpha(0)
ax.patch.set_alpha(0)

ax.bar(variances_girec.index, variances_girec.values,
       color='steelblue', edgecolor='white', linewidth=0.5)
ax.axhline(variance_threshold_girec, color='red', linewidth=1, linestyle='--',
           label=f'Threshold = {variance_threshold_girec}')

for attr, val in variances_girec.items():
    ax.text(attr, val + 0.001, f'{val:.4f}',
            ha='center', va='bottom', fontsize=12, rotation=45)

ax.set_ylabel('Variance', fontsize=12)
# ax.set_xlabel('Attribute', fontsize=12)
# ax.set_title(
#     f'Attribute variance across GIREC sub-sectors\n'
#     f'Attributes below threshold (var < {variance_threshold_girec}) highlighted in red',
#     fontsize=11
# )

for label in ax.get_xticklabels():
    if label.get_text() in low_variance_attrs_girec:
        label.set_color('#d73027')
ax.set_xticklabels(variances_girec.index, rotation=45, ha='right', fontsize=12)
ax.tick_params(axis='y', labelsize=12)

ax.spines[["top", "right"]].set_visible(False)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Étape 1 : Moyenne cantonale brute ────────────────────────────────────────
canton_means_girec = df_work_girec[attrs_girec].mean()
print(f"{'='*50}")
print(f"Canton average per attribute — GIREC sub-sectors")
print(f"{'='*50}")
for attr, val in canton_means_girec.items():
    print(f"  {attr:<25} : {val:.4f}")
print(f"{'='*50}")

In [ ]:
# ── Étape 2 : Moyenne pondérée par densité pour chaque groupe ────────────────
results_weighted_girec = []
for attr in attrs_girec:
    row = {'attribute': attr}
    for density_col in density_cols_girec:
        mask = df_work_girec[[attr, density_col]].notna().all(axis=1)
        x    = df_work_girec.loc[mask, attr]
        w    = df_work_girec.loc[mask, density_col]
        if w.sum() == 0:
            row[density_col] = np.nan
            continue
        weighted_mean    = np.average(x, weights=w)
        row[density_col] = round(weighted_mean, 4)
    results_weighted_girec.append(row)

df_weighted_girec = pd.DataFrame(results_weighted_girec).set_index('attribute')
print(f"{'='*50}")
print(f"Density-weighted means — GIREC sub-sectors")
print(f"  {df_weighted_girec.shape[0]} attributs × {df_weighted_girec.shape[1]} groupes")
print(f"{'='*50}")
display(HTML(df_weighted_girec.to_html()))

In [ ]:
# ── Étape 3 : Delta vs moyenne cantonale — GIREC ─────────────────────────────
df_delta_weighted_girec = df_weighted_girec.subtract(canton_means_girec, axis=0).round(4)
print(f"{'='*50}")
print(f"Delta — density-weighted mean vs canton average — GIREC sub-sectors")
print(f"{'='*50}")
display(HTML(df_delta_weighted_girec.to_html()))

In [ ]:
# ─── Plot delta density-weighted mean — GIREC ─────────────────────────────────
group_sizes_girec = [
    1,                      # all users
    len(gender_filters),
    len(age_filters),
    len(income_filters),
    len(car_filters),
    len(tp_filters),
]

ncols = 4
nrows = math.ceil(len(attrs_girec) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows))
fig.patch.set_alpha(0)
axes = axes.flatten()

for i, attr in enumerate(attrs_girec):
    ax     = axes[i]
    values = df_delta_weighted_girec.loc[attr]
    colors = ['#d73027' if v < 0 else '#1a9850' for v in values]
    x_labels = [
        labels_all_groups.get(c.replace('density_', ''), c.replace('density_', ''))
        for c in values.index
    ]

    ax.bar(x_labels, values.values, color=colors, edgecolor='white', linewidth=0.5)

    # ─── Transparence + fond rouge pour low variance ──────────────────────────
    if attr in low_variance_attrs_girec:
        ax.set_facecolor('#fff5f5')
        ax.patch.set_alpha(0.8)
    else:
        ax.patch.set_alpha(0)

    # ─── Séparateurs verticaux entre groupes ─────────────────────────────────
    sep_x = -0.5
    for size in group_sizes_girec[:-1]:
        sep_x += size
        ax.axvline(sep_x, color='grey', linewidth=0.8, linestyle='--', alpha=0.9)

    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylim(-0.5, 0.5)

    # ─── Titre avec classe entre parenthèses ──────────────────────────────────
    cls = attributs_info[attributs_info['attribute'] == attr]['Class'].values
    cls_label = cls[0] if len(cls) > 0 else '?'

    if attr in low_variance_attrs_girec:
        title_color = '#d73027'
        title_text  = f'{attr} ({cls_label})' #— ⚠ low variance)'
    else:
        title_color = 'black'
        title_text  = f'{attr} ({cls_label})'

    ax.set_title(title_text, fontsize=9, color=title_color, fontweight='bold')

    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=9)
    ax.tick_params(axis='y', labelsize=9)

    # ─── Valeurs numériques au dessus des barres ──────────────────────────────
    # for label, val in zip(x_labels, values.values):
    #     if val < 0:
    #         ax.text(label, val - 0.03, f'{val:.3f}',
    #                 ha='center', va='top', fontsize=7, color='#d73027', rotation=90)
    #     else:
    #         ax.text(label, val + 0.03, f'{val:.3f}',
    #                 ha='center', va='bottom', fontsize=7, color='#1a9850', rotation=90)

for j in range(i + 1, len(axes)):
    axes[j].set_axis_off()

# plt.suptitle(
#     f'Δ Density-weighted mean vs canton average by attribute | GIREC sub-sectors\n'
#     f'⚠ Red title/background = low variance attribute (var < {variance_threshold_girec})',
#     fontsize=12, y=1.01
# )
plt.tight_layout()
plt.show()

In [ ]:
# ─── Heatmap delta density-weighted mean — GIREC ──────────────────────────────
fig, ax = plt.subplots(figsize=(16, 10))
fig.patch.set_alpha(0)
ax.patch.set_alpha(0)

col_labels_girec = [
    labels_all_groups.get(c.replace('density_', ''), c.replace('density_', ''))
    for c in df_delta_weighted_girec.columns
]

im = ax.imshow(
    df_delta_weighted_girec.values,
    cmap='RdYlGn',
    aspect='auto',
    vmin=-0.6, vmax=0.6
)

ax.set_xticks(range(len(col_labels_girec)))
ax.set_yticks(range(len(attrs_girec)))
ax.set_xticklabels(col_labels_girec, rotation=35, ha='right', fontsize=12)
ax.set_yticklabels(df_delta_weighted_girec.index, fontsize=12)

# ─── Labels rouges pour les attributs sous le seuil ──────────────────────────
for label in ax.get_yticklabels():
    if label.get_text() in low_variance_attrs_girec:
        label.set_color('#d73027')

# ─── Valeurs dans les cellules ────────────────────────────────────────────────
for i in range(len(df_delta_weighted_girec.index)):
    for j in range(len(df_delta_weighted_girec.columns)):
        val = df_delta_weighted_girec.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:+.3f}',
                    ha='center', va='center', fontsize=9,
                    color='white' if abs(val) > 0.35 else 'black')

# ─── Séparateurs verticaux entre groupes socio-démo ──────────────────────────
for sep_x in [0.5, 2.5, 6.5, 9.5, 11.5]:
    ax.axvline(sep_x, color='black', linewidth=1, linestyle='--')

# ─── Séparateurs horizontaux entre classes d'attributs ────────────────────────
for sep_y in [5.5, 12.5, 17.5]:
    ax.axhline(sep_y, color='black', linewidth=1)

# ─── Colorbar ─────────────────────────────────────────────────────────────────
cbar = plt.colorbar(im, ax=ax, fraction=0.02, pad=0.01)
cbar.set_label('$\\Delta$ Attribute density-weighted mean vs canton average', fontsize=12)
cbar.ax.tick_params(labelsize=11)

# ax.set_title('Δ Density-weighted mean vs canton average — GIREC sub-sectors',
#              fontsize=12, fontweight='bold', pad=12)

plt.tight_layout()
plt.show()

In [ ]:
print(df_delta_weighted_girec.index.tolist())

In [ ]:
# ─── Density-weighted mean du walk_index par groupe ──────────────────────────
results = {}
for col in df_delta_weighted_girec.columns:  # density_homme, density_femme, etc.
    group_name = col.replace('density_', '')
    density    = zones_girec_density[col].fillna(0).values
    walk       = zones_girec_density['walk_index'].values
    
    valid = ~np.isnan(walk) & (density > 0)
    if valid.sum() > 0:
        weighted_mean = np.sum(walk[valid] * density[valid]) / np.sum(density[valid])
        canton_mean   = np.nanmean(walk)
        delta         = weighted_mean - canton_mean
        results[group_name] = {
            'weighted_mean' : weighted_mean,
            'canton_mean'   : canton_mean,
            'delta'         : delta
        }

df_walk_weighted = pd.DataFrame(results).T
print(df_walk_weighted.sort_values('delta', ascending=False).to_string())

In [ ]:
# ─── Dot plot — density-weighted walk index delta par groupe ──────────────────
df_plot = df_walk_weighted.sort_values('delta', ascending=True)

x_label_pos = df_plot['delta'].max() + 0.002  # ← position fixe pour tous les labels

fig, ax = plt.subplots(figsize=(7, 6))
fig.patch.set_alpha(0)
ax.patch.set_alpha(0)

for i, (group, row) in enumerate(df_plot.iterrows()):
    color = ALL_GROUP_COLORS.get(group, '#333333')

    # ─── Ligne depuis 0 ───────────────────────────────────────────────────────
    ax.plot([0, row['delta']], [i, i],
            color=color, linewidth=1.5, alpha=0.7)

    # ─── Point ────────────────────────────────────────────────────────────────
    ax.scatter(row['delta'], i,
               color=color, s=80, zorder=3, edgecolor='white', linewidth=0.5)

    # ─── Score aligné à droite (position fixe) ────────────────────────────────
    ax.text(x_label_pos, i, f"{row['delta']:+.3f}",
            va='center', ha='left', fontsize=12, color=color)

ax.axvline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_xlim(left=-0.002, right=x_label_pos + 0.012)
ax.set_yticks(range(len(df_plot)))
ax.set_yticklabels([labels_all_groups.get(g, g) for g in df_plot.index], fontsize=11)
ax.set_xlabel('$\\Delta$ density-weighted walk index vs canton average', fontsize=12)
ax.tick_params(axis='x', labelsize=11)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## DELTA MEAN BY AREA TYPE - GIREC

In [ ]:
# ── Vérification AREA_TYPE sur GIREC ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))

for area_type, color in AREA_TYPE_COLORS.items():
    mask = zones_girec_density['AREA_TYPE'].str.lower() == area_type
    if mask.sum() > 0:
        zones_girec_density[mask].plot(ax=ax, color=color, linewidth=0.3, 
                                        edgecolor='white', alpha=0.85)

ax.set_title('AREA_TYPE — GIREC sub-sectors', fontsize=12, fontweight='bold')
ax.set_axis_off()

patches = [mpatches.Patch(color=AREA_TYPE_COLORS[a], label=a.title())
           for a in AREA_TYPE_ORDER]
ax.legend(handles=patches, loc='lower left', fontsize=9, framealpha=0.9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Analyse stratifiée par AREA_TYPE — GIREC ──────────────────────────────────
df_work_area_girec = df_work_girec.copy()
df_work_area_girec['AREA_TYPE'] = zones_girec_density['AREA_TYPE'].str.lower()

results_by_area_girec = {}

for area_type in AREA_TYPE_ORDER:
    df_area_girec = df_work_area_girec[
        df_work_area_girec['AREA_TYPE'] == area_type
    ].drop(columns='AREA_TYPE')

    if len(df_area_girec) == 0:
        continue

    rows = []
    for attr in attrs_girec:
        row = {'attribute': attr}
        for density_col in density_cols_girec:
            mask = df_area_girec[[attr, density_col]].notna().all(axis=1)
            x    = df_area_girec.loc[mask, attr]
            w    = df_area_girec.loc[mask, density_col]
            if w.sum() == 0:
                row[density_col] = np.nan
                continue
            row[density_col] = round(np.average(x, weights=w), 4)
        rows.append(row)

    df_weighted_area_girec = pd.DataFrame(rows).set_index('attribute')
    df_delta_area_girec    = df_weighted_area_girec.subtract(
        canton_means_girec, axis=0).round(4)

    results_by_area_girec[area_type] = {
        'weighted': df_weighted_area_girec,
        'delta':    df_delta_area_girec,
        'n':        len(df_area_girec)
    }

    print(f"\n{'='*50}")
    print(f"  {area_type.title()} (n={len(df_area_girec)} sous-secteurs GIREC)")
    print(f"{'='*50}")
    display(HTML(df_delta_area_girec.to_html()))

In [ ]:
# ── Heatmap par AREA_TYPE — GIREC ─────────────────────────────────────────────
col_labels_girec = [
    labels_all_groups.get(c.replace('density_', ''), c.replace('density_', ''))
    for c in density_cols_girec
]

for area_type in AREA_TYPE_ORDER:
    if area_type not in results_by_area_girec:
        continue

    df_delta_area_girec = results_by_area_girec[area_type]['delta']
    n_girec             = results_by_area_girec[area_type]['n']

    fig, ax = plt.subplots(figsize=(16, 10))

    im = ax.imshow(
        df_delta_area_girec.values,
        cmap='RdYlGn',
        aspect='auto',
        vmin=-0.6, vmax=0.6
    )

    ax.set_xticks(range(len(col_labels_girec)))
    ax.set_yticks(range(len(df_delta_area_girec.index)))
    ax.set_xticklabels(col_labels_girec, rotation=35, ha='right', fontsize=9)
    ax.set_yticklabels(df_delta_area_girec.index, fontsize=9)

    for i in range(len(df_delta_area_girec.index)):
        for j in range(len(df_delta_area_girec.columns)):
            val = df_delta_area_girec.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f'{val:+.3f}',
                        ha='center', va='center', fontsize=7,
                        color='white' if abs(val) > 0.35 else 'black')

    plt.colorbar(im, ax=ax, fraction=0.02, pad=0.01,
                 label='Δ density-weighted mean vs canton average')

    ax.set_title(
        f'Δ Density-weighted mean vs canton average\n'
        f'{area_type.title()} (n={n_girec} sous-secteurs GIREC)',
        fontsize=12, fontweight='bold', pad=12
    )

    plt.tight_layout()
    plt.show()

In [ ]:
# ─── Lignes par AREA_TYPE — variabilité inter-groupes selon le type de zone ───
ncols = 4
nrows = math.ceil(len(attrs_girec) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = axes.flatten()

x_labels = [
    labels_all_groups.get(c.replace('density_', ''), c.replace('density_', ''))
    for c in density_cols_girec
]
x_pos = range(len(x_labels))

for i, attr in enumerate(attrs_girec):
    ax = axes[i]

    for area_type in AREA_TYPE_ORDER:
        if area_type not in results_by_area_girec:
            continue
        values = results_by_area_girec[area_type]['delta'].loc[attr].values
        ax.plot(x_pos, values, marker='o', markersize=4,
                color=AREA_TYPE_COLORS[area_type],
                label=area_type.title(), linewidth=1.5, alpha=0.85)

    ax.axhline(0, color='black', linewidth=0.8, linestyle='-')
    ax.set_ylim(-0.6, 0.6)
    ax.set_title(attr, fontsize=9,
                 color='#d73027' if attr in low_variance_attrs_girec else 'black')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=6)
    ax.spines[["top", "right"]].set_visible(False)

    if attr in low_variance_attrs_girec:
        ax.set_facecolor('#fff5f5')

# Légende commune
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=len(AREA_TYPE_ORDER),
           bbox_to_anchor=(0.5, 1.02), fontsize=10)

for j in range(i + 1, len(axes)):
    axes[j].set_axis_off()

plt.suptitle(
    'Δ Density-weighted mean vs canton average by attribute and group\n'
    'One line per AREA_TYPE — GIREC sub-sectors',
    fontsize=12, y=1.04
)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Print des résultats de variabilité inter-groupes par AREA_TYPE ───────────
print(f"\n{'='*70}")
print(f"Inter-group std dev of density-weighted attribute means by AREA_TYPE")
print(f"{'='*70}")

results_std = {}
for area_type in AREA_TYPE_ORDER:
    if area_type not in results_by_area_girec:
        continue
    dispersion_vals = [
        results_by_area_girec[area_type]['weighted'].loc[attr].std()
        for attr in attrs_girec
    ]
    results_std[area_type] = dispersion_vals

df_std = pd.DataFrame(results_std, index=attrs_girec)
df_std.columns = [c.title() for c in df_std.columns]

print(f"\n{'Attribute':<25}", end='')
for col in df_std.columns:
    print(f" {col[:20]:>20}", end='')
print()
print(f"{'-'*70}")
for attr in attrs_girec:
    print(f"  {attr:<23}", end='')
    for col in df_std.columns:
        print(f" {df_std.loc[attr, col]:>20.4f}", end='')
    print()

print(f"\n{'─'*70}")
print(f"{'Mean std dev':<25}", end='')
for col in df_std.columns:
    print(f" {df_std[col].mean():>20.4f}", end='')
print()
print(f"{'='*70}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
fig.patch.set_alpha(0)
ax.patch.set_alpha(0)

for area_type in AREA_TYPE_ORDER:
    if area_type not in results_by_area_girec:
        continue
    if area_type == 'major metro centers':
        continue
    dispersion_vals = [
        results_by_area_girec[area_type]['weighted'].loc[attr].std()  # ← weighted au lieu de delta
        for attr in attrs_girec
    ]
    ax.plot(attrs_girec, dispersion_vals, marker='o', markersize=6,
            color=AREA_TYPE_COLORS[area_type],
            label=AREA_TYPE_LABELS.get(area_type, area_type),  # ← mapping labels
            linewidth=2.5, alpha=1, zorder=2)

if 'major metro centers' in results_by_area_girec:
    dispersion_vals = [
        results_by_area_girec['major metro centers']['weighted'].loc[attr].std()
        for attr in attrs_girec
    ]
    ax.plot(attrs_girec, dispersion_vals, marker='o', markersize=6,
            color=AREA_TYPE_COLORS['major metro centers'],
            label=AREA_TYPE_LABELS.get('major metro centers', 'Major metro centers'),  # ← mapping labels
            linewidth=2.5, alpha=1.0, zorder=10)

ax.set_xticks(range(len(attrs_girec)))
ax.set_xticklabels(attrs_girec, rotation=45, ha='right', fontsize=12)
ax.tick_params(axis='y', labelsize=12)
ax.set_ylabel('Std dev of weighted mean \n across socio-demo groups', fontsize=12)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(fontsize=12, loc='upper center', bbox_to_anchor=(0.5, -0.35),
          ncol=3, frameon=False)
# ax.set_title(
#     'Inter-group variability of density-weighted attribute means\n'
#     'by area type',
#     fontsize=13, fontweight='bold'
# )
plt.tight_layout()
plt.show()

In [ ]:
# ─── Densité moyenne par groupe et par AREA_TYPE ──────────────────────────────
df_density_area = zones_girec_density.copy()
df_density_area['AREA_TYPE'] = df_density_area['AREA_TYPE'].str.lower()

# Moyenne de chaque colonne density_* par AREA_TYPE
density_by_area = (
    df_density_area
    .groupby('AREA_TYPE')[density_cols_girec]
    .mean()
    .reindex(AREA_TYPE_ORDER)
)

print(density_by_area.to_string())

# ─── Plot ─────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))

x       = np.arange(len(AREA_TYPE_ORDER))
n_groups = len(density_cols_girec)
width   = 0.8 / n_groups

for i, density_col in enumerate(density_cols_girec):
    label = labels_all_groups.get(
        density_col.replace('density_', ''), 
        density_col.replace('density_', '')
    )
    ax.bar(
        x + i * width - 0.4 + width/2,
        density_by_area[density_col].values,
        width=width,
        label=label
    )

ax.set_xticks(x)
ax.set_xticklabels([a.title() for a in AREA_TYPE_ORDER], rotation=20, ha='right')
ax.set_ylabel('Mean pedestrian density\n[legs · day⁻¹ · user⁻¹]')
ax.set_title(
    'Mean pedestrian density by group and area type',
    fontsize=13, fontweight='bold'
)
ax.legend(fontsize=8, ncol=4, loc='upper right')
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

# TEST ATTRIBUTS

In [ ]:
def plot_walk_attribute(gdf, attribute, canton_GE=None, girec=None,
                        cmap='RdYlGn', figsize=(12, 12),
                        vmin=None, vmax=None, title=None):
    """
    Plot un attribut du walk_index à l'échelle cantonale.
    
    Parameters
    ----------
    gdf       : GeoDataFrame → index_walkability
    attribute : str          → nom de la colonne à visualiser
    canton_GE : GeoDataFrame → contour du canton (optionnel)
    girec     : GeoDataFrame → contour des sous-secteurs GIREC (optionnel)
    cmap      : str          → colormap (défaut RdYlGn)
    figsize   : tuple        → taille de la figure
    vmin      : float        → valeur min de la colormap (défaut : min de l'attribut)
    vmax      : float        → valeur max de la colormap (défaut : max de l'attribut)
    title     : str          → titre de la figure (défaut : nom de l'attribut)
    """
    if attribute not in gdf.columns:
        raise ValueError(f"Attribut '{attribute}' introuvable dans le GeoDataFrame")

    # ─── Valeurs min/max ──────────────────────────────────────────────────────
    vmin = vmin if vmin is not None else gdf[attribute].quantile(0.01)
    vmax = vmax if vmax is not None else gdf[attribute].quantile(0.99)

    # ─── Plot ─────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')

    gdf.plot(
        ax=ax,
        column=attribute,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        linewidth=0.3,
        alpha=1.0,
        zorder=1,
        legend=True,
        legend_kwds={
            'label'      : attribute,
            'fraction'   : 0.03,
            'pad'        : 0.01,
            'shrink'     : 0.8,
        }
    )

    # ─── Boundaries optionnelles ──────────────────────────────────────────────
    if girec is not None:
        girec.to_crs(gdf.crs).boundary.plot(
            ax=ax, color='gray', linewidth=0.4, alpha=0.4, zorder=2)

    if canton_GE is not None:
        canton_GE.to_crs(gdf.crs).boundary.plot(
            ax=ax, color='black', linewidth=1.5, zorder=3)

    ax.set_axis_off()
    ax.set_title(
        title if title is not None else f'Walk index attribute — {attribute}',
        fontsize=13, fontweight='bold', pad=15
    )

    plt.tight_layout()
    plt.show()

In [ ]:
# Exemple sur un attribut
plot_walk_attribute(
    gdf       = index_walkability,
    attribute = 'pente',
    canton_GE = canton_GE,
    girec     = None
)

# Avec colormap divergente pour les attributs négatifs
plot_walk_attribute(
    gdf       = index_walkability,
    attribute = 'bruit',
    canton_GE = canton_GE,
    girec     = None,
    cmap      = 'RdYlGn'  # inversé car attribut négatif
)

In [ ]:
plot_walk_attribute(
    gdf       = index_walkability,
    attribute = 'fontaine',
    canton_GE = canton_GE,
    girec     = None
)